# Model Training

Train the Random Forest classifier:
1. Prepare preprocessed data
2. Initialize model
3. Train model
4. Extract feature importance
5. Save model

## Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Add project to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from src import config
from src.data_preprocessing import preprocess_data
from src.train import train_model, get_feature_importance, save_model

print("✓ Imports complete")

## Step 1: Load Preprocessed Data

In [ ]:
# Preprocess data
X_train, X_test, y_train, y_test = preprocess_data(project_root / config.RAW_DATA_PATH)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"\nFeature names: {X_train.columns.tolist()}")

## Step 2: Model Configuration

In [ ]:
# Display model configuration
print("Random Forest Configuration:")
print(f"  n_estimators: {config.N_ESTIMATORS}")
print(f"  max_depth: {config.RANDOM_FOREST_MAX_DEPTH}")
print(f"  random_state: {config.RANDOM_STATE}")
print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## Step 3: Train Model

In [ ]:
# Train the model
model = train_model(
    X_train, y_train,
    n_estimators=config.N_ESTIMATORS,
    max_depth=config.RANDOM_FOREST_MAX_DEPTH,
    random_state=config.RANDOM_STATE
)

print(f"\n✓ Model trained successfully!")
print(f"\nModel details:")
print(f"  Number of trees: {model.n_estimators}")
print(f"  Max depth: {model.max_depth}")
print(f"  Classes: {model.classes_}")

## Step 4: Model Performance on Training Data

In [ ]:
# Get training accuracy
train_score = model.score(X_train, y_train)

print(f"Training Accuracy: {train_score:.4f}")

# Make predictions
y_train_pred = model.predict(X_train)

# Count correct predictions
correct = (y_train_pred == y_train).sum()
total = len(y_train)

print(f"\nCorrect predictions: {correct}/{total}")

## Step 5: Extract Feature Importance

In [ ]:
# Get feature importance
feature_importance = get_feature_importance(
    model,
    X_train.columns.tolist(),
    top_n=10
)

# Display all features
print("\nAll Features (by importance):")
print(feature_importance.to_string(index=False))

## Step 6: Visualize Feature Importance

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))

# Top features
top_n = 10
top_features = feature_importance.head(top_n)

ax.barh(range(len(top_features)), top_features['importance'])
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.set_xlabel('Importance Score')
ax.set_title(f'Top {top_n} Most Important Features')
ax.invert_yaxis()

# Add value labels
for i, v in enumerate(top_features['importance']):
    ax.text(v + 0.01, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.show()

print("✓ Feature importance visualization complete!")

## Step 7: Save Model

In [ ]:
# Save model
save_model(model, project_root / config.MODEL_PATH)

# Verify model was saved
model_path = project_root / config.MODEL_PATH
if model_path.exists():
    file_size = model_path.stat().st_size / 1024  # Size in KB
    print(f"✓ Model saved successfully!")
    print(f"  Path: {model_path}")
    print(f"  Size: {file_size:.2f} KB")
else:
    print("❌ Error saving model!")

## Step 8: Load and Verify Saved Model

In [ ]:
# Load the saved model
loaded_model = joblib.load(project_root / config.MODEL_PATH)

print("✓ Model loaded successfully!")
print(f"\nVerify loaded model matches original:")
print(f"  Same n_estimators? {loaded_model.n_estimators == model.n_estimators}")
print(f"  Same max_depth? {loaded_model.max_depth == model.max_depth}")
print(f"  Same classes? {np.array_equal(loaded_model.classes_, model.classes_)}")

# Test prediction with loaded model
test_sample = X_train.iloc[0:1]
pred_original = model.predict(test_sample)
pred_loaded = loaded_model.predict(test_sample)

print(f"\n  Same predictions? {np.array_equal(pred_original, pred_loaded)}")
print(f"\n✓ Saved model verified!")

## Summary

✓ Model training complete!

### Training Summary:
- Model type: Random Forest Classifier
- Number of trees: {{config.N_ESTIMATORS}}
- Max depth: {{config.RANDOM_FOREST_MAX_DEPTH}}
- Training accuracy: {{train_score:.4f}}
- Model saved: {{model_path}}

### Next Steps:
Continue to `05_model_evaluation.ipynb` to evaluate model performance on test data.